In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from numba import njit
import time


# ============================================================
# Thermal Model 
# Inputs here are power dissipation (P) in electronics in Watts and temperature (S) boundary conditions in degree Celcius.
# The temperature boundary conditions are a function of external heat sources. For instance, if there is a heat source in close proximity to the electronics,
# and the effect of the heat source in the thermal system is observed / measured in terms of temperature rise in the modeled thermal system. Eventhough the overall
# mathematical formulation remains the same, the two sources are treated separately because they have different units. Thus the final form of transient temperature 
# equation is expressed as:
# T_i (t) = T_0 + \sum_{j=1} ^{N_P} \sum_m \Delta (P_j R_{ij})_m (1-exp^{-K_{ij}(t-t_m)} + \sum_{k=1} ^{N_S} \sum_m \Delta (T_S_k G_{ik})_m (1-exp^{-L_{ik}(t-t_m)}
# 
# This separation of sources allows to preserve the physical meaning
# -----------------------------------------------------------------------
#  Input                 |    Gain Matrix          |     Time Constant
# Power Dissipation (W)  |       R [C/W]           |      K = 1/RC [1/s]
# Temperature source (C) |       G [C/C ]          |      L [1/s]
# ============================================================

@njit(cache=True, fastmath=True)
def thermal_model_vec(P,S,t,R_row,K_row,G_row,L_row,T0):

    N_power, T_len = P.shape
    N_sources = S.shape[0]

    T_out = np.zeros(T_len)

    # ===================================================
    # POWER INPUTS
    # ===================================================

    for j in range(N_power):

        Rj = R_row[j]
        Kj = K_row[j]

        Pj = P[j].copy()
        Pj[0] = 0.0

        idx_list = []

        for k in range(1, T_len):

            if Pj[k] != Pj[k - 1]:
                idx_list.append(k)

        if len(idx_list) == 0:
            continue

        TC = Pj * Rj

        for idx in idx_list:

            delta = TC[idx] - TC[idx - 1]
            t0 = t[idx - 1]

            dt = t[idx:] - t0

            T_out[idx:] += (delta * (1.0 - np.exp(-Kj * dt)))

    # ===================================================
    # TEMPERATURE SOURCE INPUTS
    # ===================================================

    for s in range(N_sources):

        Gs = G_row[s]
        Ls = L_row[s]

        Ts = S[s].copy()

        idx_list = []

        for k in range(1, T_len):

            if Ts[k] != Ts[k - 1]:
                idx_list.append(k)

        if len(idx_list) == 0:
            continue

        TC = Ts * Gs

        for idx in idx_list:

            delta = TC[idx] - TC[idx - 1]
            t0 = t[idx - 1]

            dt = t[idx:] - t0

            T_out[idx:] += (delta * (1.0 - np.exp(-Ls * dt)))

    return T0 + T_out


# ============================================================
# Rank Reduced Factors
# ============================================================

def unpack_rank_factors(theta,n_temps,n_power,n_sources,r_power,r_source):

    idx = 0

    # ==========================================
    # R MATRIX
    # ==========================================

    AR = theta[idx:idx+n_temps*r_power].reshape(n_temps,r_power)
    idx += n_temps*r_power

    BR = theta[idx:idx+n_power*r_power].reshape(n_power,r_power)
    idx += n_power*r_power

    # ==========================================
    # K MATRIX
    # ==========================================

    AK = theta[idx:idx+n_temps*r_power].reshape(n_temps,r_power)
    idx += n_temps*r_power

    BK = theta[idx:idx+n_power*r_power].reshape(n_power,r_power)
    idx += n_power*r_power

    # ==========================================
    # G MATRIX
    # ==========================================

    AG = theta[idx:idx+n_temps*r_source].reshape(n_temps,r_source)
    idx += n_temps*r_source

    BG = theta[idx:idx+n_sources*r_source].reshape(n_sources,r_source)
    idx += n_sources*r_source

    # ==========================================
    # L MATRIX
    # ==========================================

    AL = theta[idx:idx+n_temps*r_source].reshape(n_temps,r_source)
    idx += n_temps*r_source

    BL = theta[idx:].reshape(n_sources,r_source)

    R = AR @ BR.T
    K = AK @ BK.T

    G = AG @ BG.T
    L = AL @ BL.T

    return R, K, G, L


# ============================================================
# Residual Function
# ============================================================

def residuals_rank(theta,t,P,S,T_meas,T0,n_temps,n_power,n_sources,r_power,r_source):

    R, K, G, L = unpack_rank_factors(theta,n_temps,n_power,n_sources,r_power,r_source)

    T_pred = np.zeros_like(T_meas)

    for i in range(n_temps):

        T_pred[i] = thermal_model_vec(P,S,t,R[i],K[i],G[i],L[i],T0)

    return (T_pred - T_meas).ravel()


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    # ========================================================
    # Load Excel sheets
    # ========================================================

    power_df = pd.read_excel("Training_Data_1.xlsx",sheet_name="PowerDissipation_8s")

    temp_df = pd.read_excel("Training_Data_1.xlsx",sheet_name="Temperatures_8s")

    # ========================================================
    # Power Columns
    # ========================================================

    P_cols = [
    "P_Q2000",
    "P_Q2001",
    "P_Q2002",
    "P_Q2003",
    "P_Q2004",
    "P_Q2005",
    "P_R2016",
    "P_L2000",
    "P_Q1500",
    "P_Q1501",
    "P_Q11504",
    "P_Q11505",
    "P_C2012",
    "P_C2013",
    "P_C2077",
    "P_IC20000",
    "P_IC9000",
    "P_IC10601",
    "P_IC31900"
    ]
    
    S_cols = [
        "T_S1",
        "T_S2",
        "T_S3"
    ]

    # ========================================================
    # Temperature Columns
    # ========================================================

    T_cols = [
        "T_Q2000_Junction",
        "T_Q2000_Case",
        "T_Q2001_Junction",
        "T_Q2001_Case",
        "T_Q2002_Junction",
        "T_Q2002_Case",
        "T_Q2003_Junction",
        "T_Q2003_Case",
        "T_Q2004_Junction",
        "T_Q2004_Case",
        "T_Q2005_Junction",
        "T_Q2005_Case",
        "T_R2016",
        "T_L2000_Source",
        "T_L2000_Case",
        "T_Q1500_Junction",
        "T_Q1500_Case",
        "T_Q1501_Junction",
        "T_Q1501_Case",
        "T_Q11504_Junction",
        "T_Q11504_Case",
        "T_Q11505_Junction",
        "T_Q11505_Case",
        "T_C2012_Src",
        "T_C2012_Case",
        "T_C2013_Src",
        "T_C2013_Case",
        "T_C2077_Src",
        "T_C2077_Case",
        "T_IC20000_Src",
        "T_IC20000_Case",
        "T_IC9000_Src",
        "T_IC9000_Case",
        "T_IC10601_Src",
        "T_IC10601_Case",
        "T_IC31900",
        "T_PCBA"
    ]

    # ========================================================
    # Time consistency check
    # ========================================================

    t_power = power_df["t"].values
    t_temp = temp_df["t"].values

    if not np.array_equal(t_power, t_temp):
        raise ValueError("Time vectors in PowerDissipation and Temperatures do not match.")

    t = t_power

    # ========================================================
    # Build matrices
    # ========================================================

    P = np.vstack([power_df[col].values for col in P_cols])

    S = np.vstack([power_df[col].values for col in S_cols])
    
    T_meas = np.vstack([temp_df[col].values for col in T_cols])


    n_power = P.shape[0]
    n_sources = S.shape[0]
    
    n_temps = T_meas.shape[0]

    print(f"Power inputs      : {n_power}")
    print(f"Temperature sources      : {n_sources}")
    print(f"Temperature monitor points : {n_temps}")

    T0 = 20.0

    # ========================================================
    # Rank selection
    # ========================================================

    r = 3
    r_power = 3
    r_source = 3

    n_params = (2 * (n_temps + n_power) * r_power  +  2 * (n_temps + n_sources) * r_source)

    print(f"\nRank={r}  ->  {n_params} optimization parameters")

    # ========================================================
    # Initial Guess
    # ========================================================

    theta0 = np.ones(n_params) * 0.1

    lower = np.zeros_like(theta0)
    upper = np.full_like(theta0, np.inf)

    # ========================================================
    # Optimization
    # ========================================================

    print("\n===================================")
    print("Starting Parameter Estimation")
    print("===================================")

    start = time.time()

    result = least_squares(
        residuals_rank,
        theta0,
        args=(
            t,
            P,
            S,
            T_meas,
            T0,
            n_temps,
            n_power,
            n_sources,
            r_power,
            r_source
        ),
        bounds=(lower, upper),
        method="trf",
        verbose=2,
        max_nfev=30000
    )

    elapsed = time.time() - start

    print(
        f"\nOptimization completed in {elapsed:.2f} sec"
    )

    # ========================================================
    # Extract Final Matrices
    # ========================================================

    R_est, K_est, G_est, L_est = unpack_rank_factors(result.x,n_temps,n_power,n_sources,r_power,r_source)
    print("\nR matrix shape:", R_est.shape)
    print("K matrix shape:", K_est.shape)

    # ========================================================
    # Save Parameters
    # ========================================================

    rows = []
    
    # ==========================================
    # R Matrix
    # ==========================================
    
    for i in range(n_temps):
        for j in range(n_power):
    
            rows.append([f"R_{i+1}_{j+1}",R_est[i, j]])
    
    # ==========================================
    # K Matrix
    # ==========================================
    
    for i in range(n_temps):
        for j in range(n_power):
    
            rows.append([f"K_{i+1}_{j+1}",K_est[i, j]])
    
    # ==========================================
    # G Matrix
    # ==========================================
    
    for i in range(n_temps):
        for j in range(n_sources):
    
            rows.append([f"G_{i+1}_{j+1}",G_est[i, j]])
    
    # ==========================================
    # L Matrix
    # ==========================================
    
    for i in range(n_temps):
        for j in range(n_sources):
    
            rows.append([f"L_{i+1}_{j+1}",L_est[i, j]])

    # ========================================================
    # Optional: Save R and K as matrices
    # ========================================================

    with pd.ExcelWriter("ROM_Matrices.xlsx",engine="openpyxl") as writer:
    
        pd.DataFrame(R_est,index=T_cols,columns=P_cols).to_excel(writer,sheet_name="R_Matrix")
    
        pd.DataFrame(K_est,index=T_cols,columns=P_cols).to_excel(writer,sheet_name="K_Matrix")
    
        pd.DataFrame(G_est,index=T_cols,columns=S_cols).to_excel(writer,sheet_name="G_Matrix")
    
        pd.DataFrame(L_est,index=T_cols,columns=S_cols).to_excel(writer,sheet_name="L_Matrix")

    print("Saved: RK_Matrices.xlsx")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit


# ============================================================
# Temperature Names
# ============================================================

TEMPERATURE_NAMES = [
    "T_Q2000_Junction",
    "T_Q2000_Case",
    "T_Q2001_Junction",
    "T_Q2001_Case",
    "T_Q2002_Junction",
    "T_Q2002_Case",
    "T_Q2003_Junction",
    "T_Q2003_Case",
    "T_Q2004_Junction",
    "T_Q2004_Case",
    "T_Q2005_Junction",
    "T_Q2005_Case",
    "T_R2016",
    "T_L2000_Source",
    "T_L2000_Case",
    "T_Q1500_Junction",
    "T_Q1500_Case",
    "T_Q1501_Junction",
    "T_Q1501_Case",
    "T_Q11504_Junction",
    "T_Q11504_Case",
    "T_Q11505_Junction",
    "T_Q11505_Case",
    "T_C2012_Src",
    "T_C2012_Case",
    "T_C2013_Src",
    "T_C2013_Case",
    "T_C2077_Src",
    "T_C2077_Case",
    "T_IC20000_Src",
    "T_IC20000_Case",
    "T_IC9000_Src",
    "T_IC9000_Case",
    "T_IC10601_Src",
    "T_IC10601_Case",
    "T_IC31900",
    "T_PCBA"
]


# ============================================================
# Power Channels
# ============================================================

POWER_COLUMNS = [
    "P_Q2000",
    "P_Q2001",
    "P_Q2002",
    "P_Q2003",
    "P_Q2004",
    "P_Q2005",
    "P_R2016",
    "P_L2000",
    "P_Q1500",
    "P_Q1501",
    "P_Q11504",
    "P_Q11505",
    "P_C2012",
    "P_C2013",
    "P_C2077",
    "P_IC20000",
    "P_IC9000",
    "P_IC10601",
    "P_IC31900"
]

TEMPERATURE_SOURCE_COLUMNS = [
    "T_S1",
    "T_S2",
    "T_S3"
]


# ============================================================
# Load Validation Data
# ============================================================

def load_validation_inputs(filename):

    df = pd.read_excel(filename,sheet_name="PowerDissipation_11s")

    t = df["t"].values

    P = np.vstack([df[col].values for col in POWER_COLUMNS])

    S = np.vstack([df[col].values for col in TEMPERATURE_SOURCE_COLUMNS])

    return t, P, S


# ============================================================
# Load Parameter File
# Expects:
# R_1_1 ... R_37_19
# K_1_1 ... K_37_19
# ============================================================

def load_ROM_matrices(filename):

    R = pd.read_excel(filename,sheet_name="R_Matrix",index_col=0).values

    K = pd.read_excel(filename,sheet_name="K_Matrix",index_col=0).values

    G = pd.read_excel(filename,sheet_name="G_Matrix",index_col=0).values

    L = pd.read_excel(filename,sheet_name="L_Matrix",index_col=0).values

    return R, K, G, L


# ============================================================
# Thermal Model
# ============================================================

@njit(cache=True, fastmath=True)
def thermal_model_vec(P,S,t,R_row,K_row,G_row,L_row,T0):

    N_power, T_len = P.shape
    N_sources = S.shape[0]

    T_out = np.zeros(T_len)

    # ===========================================
    # POWER INPUTS
    # ===========================================

    for j in range(N_power):

        Rj = R_row[j]
        Kj = K_row[j]

        Pj = P[j].copy()
        Pj[0] = 0.0

        TC = Pj * Rj

        for k in range(1, T_len):

            if Pj[k] == Pj[k - 1]:
                continue

            delta = TC[k] - TC[k - 1]

            t0 = t[k - 1]

            for n in range(k, T_len):

                dt = t[n] - t0

                T_out[n] += (delta * (1.0 - np.exp(-Kj * dt)))

    # ===========================================
    # TEMPERATURE SOURCES
    # ===========================================

    for s in range(N_sources):

        Gs = G_row[s]
        Ls = L_row[s]

        Ts = S[s]

        TC = Ts * Gs

        for k in range(1, T_len):

            if Ts[k] == Ts[k - 1]:
                continue

            delta = TC[k] - TC[k - 1]

            t0 = t[k - 1]

            for n in range(k, T_len):

                dt = t[n] - t0

                T_out[n] += (delta * (1.0 - np.exp(-Ls * dt)))

    return T0 + T_out


# ============================================================
# Compute All Temperatures
# ============================================================

def compute_temperatures(P,S,t,R,K,G,L,T0=20.0):

    n_temps = R.shape[0]

    T_pred = np.zeros((n_temps, len(t)))

    for i in range(n_temps):

        T_pred[i] = thermal_model_vec(P,S,t,R[i],K[i],G[i],L[i],T0)

    return T_pred


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    validation_file = "Validation2.xlsx"
    #parameter_file = "parameters_rank_reduced.xlsx"
    parameter_file = "ROM_Matrices.xlsx"

    print("\nLoad validation data...")

    t, P, S = load_validation_inputs(validation_file)

    print(f"Power matrix shape : {P.shape}")
    
    print(f"Temperature Source matrix shape: {S.shape}")

    print("\nLoad parameters...")

    R, K, G, L = load_ROM_matrices(parameter_file)

    print(f"R shape : {R.shape}")
    print(f"K shape : {K.shape}")
    print(f"G shape : {G.shape}")
    print(f"L shape : {L.shape}")


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    print("\nCompute temperatures...")

    T_pred = compute_temperatures(P,S,t,R,K,G,L,T0=20.0)
    print(f"Generated {T_pred.shape[0]} temperature channels")

    # --------------------------------------------------------
    # Save Results
    # --------------------------------------------------------

    df_out = pd.DataFrame()
    df_out["t"] = t

    for i in range(T_pred.shape[0]):

        if i < len(TEMPERATURE_NAMES):
            column_name = TEMPERATURE_NAMES[i]
        else:
            column_name = f"T_{i+1}"

        df_out[column_name] = T_pred[i]

    output_file = "Predicted_Temperatures.xlsx"

    df_out.to_excel(output_file,index=False)

    print(f"\nSaved: {output_file}")

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plt.figure(figsize=(16, 9))

    for i in range(T_pred.shape[0]):

        if i < len(TEMPERATURE_NAMES):
            label = TEMPERATURE_NAMES[i]
        else:
            label = f"T_{i+1}"

        plt.plot(t,T_pred[i],label=label,linewidth=1)

    plt.title("Predicted Temperatures")
    plt.xlabel("Time [s]")
    plt.ylabel("Temperature [°C]")
    plt.grid(True)

    plt.legend(fontsize=7,ncol=3)

    plt.tight_layout()
    plt.show()